In [1]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv("C:\\Users\\Anuj Kumar\\Desktop\\Machine Learning\\datasets\\cardekho_imputated.csv")

In [5]:
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [6]:
df.drop(["Unnamed: 0","car_name","model"],axis=1,inplace=True)

In [7]:
df.columns

Index(['brand', 'vehicle_age', 'km_driven', 'seller_type', 'fuel_type',
       'transmission_type', 'mileage', 'engine', 'max_power', 'seats',
       'selling_price'],
      dtype='str')

In [9]:
for i in df.columns:
    if df[i].dtype == "str":
        print(i,df[i].unique())

brand <StringArray>
[       'Maruti',       'Hyundai',          'Ford',       'Renault',
          'Mini', 'Mercedes-Benz',        'Toyota',    'Volkswagen',
         'Honda',      'Mahindra',        'Datsun',          'Tata',
           'Kia',           'BMW',          'Audi',    'Land Rover',
        'Jaguar',            'MG',         'Isuzu',       'Porsche',
         'Skoda',         'Volvo',         'Lexus',          'Jeep',
      'Maserati',       'Bentley',        'Nissan',         'ISUZU',
       'Ferrari',  'Mercedes-AMG',   'Rolls-Royce',         'Force']
Length: 32, dtype: str
seller_type <StringArray>
['Individual', 'Dealer', 'Trustmark Dealer']
Length: 3, dtype: str
fuel_type <StringArray>
['Petrol', 'Diesel', 'CNG', 'LPG', 'Electric']
Length: 5, dtype: str
transmission_type <StringArray>
['Manual', 'Automatic']
Length: 2, dtype: str


In [10]:
df.isna().sum()

brand                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [11]:
X  = df.drop("selling_price",axis=1)
y = df["selling_price"]

In [12]:
X.head(5)

,brand,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Maruti,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,Hyundai,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,Hyundai,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,Maruti,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,Ford,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [16]:
num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(include="object").columns

C:\Users\Anuj Kumar\AppData\Local\Temp\ipykernel_20588\3822978638.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns


In [17]:
num_cols

Index(['vehicle_age', 'km_driven', 'mileage', 'engine', 'max_power', 'seats'], dtype='str')

In [19]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [20]:
from sklearn.preprocessing import OneHotEncoder,StandardScaler,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline   
from sklearn.ensemble import GradientBoostingRegressor

In [21]:
label = ["brand"]
onehot = [col for col in cat_cols if col not in label]

In [26]:
preprocess = ColumnTransformer([
    ("onehot", OneHotEncoder(
        sparse_output=False,
        drop="first",
        handle_unknown="ignore"
    ), cat_cols),

    ("scaler", StandardScaler(), num_cols)
])

In [27]:
model = GradientBoostingRegressor(n_estimators=100,learning_rate=0.1,max_depth=3,random_state=42)


In [28]:
pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", model)    
])

In [29]:
pipeline.fit(x_train,y_train)

predictions = pipeline.predict(x_test)

c:\Users\Anuj Kumar\Desktop\Machine Learning\.env\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [30]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)
mape = mean_absolute_percentage_error(y_test, predictions) * 100

print("Mean Absolute Error (MAE):", mae)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("R² Score:", r2)
print("Mean Absolute Percentage Error (MAPE):", mape, "%")

Mean Absolute Error (MAE): 125211.05260971202
Mean Squared Error (MSE): 57773611356.245865
Root Mean Squared Error (RMSE): 240361.41819403102
R² Score: 0.9232531763033732
Mean Absolute Percentage Error (MAPE): 17.799463668855967 %
